# Seishin Vision Agent — multimodal on-device tool-calling model

Fine-tunes **Qwen3-VL-2B-Instruct** so Seishin's on-device agent can *see* the photos you attach —
posters, bills, assignment sheets, class schedules, chat screenshots, handwritten notes — and turn
them into events, tasks and notes with the app's real tools (`add_event`, `add_todo`, `add_note`, …),
while keeping all the text-only tool calling.

**How to run:** `Runtime → Change runtime type → GPU` (T4 works; L4/A100 is ~3× faster), then `Runtime → Run all`.

1. Renders ~1.2k synthetic photos + conversations (plus text-only chats) from the app's real tool schemas
2. LoRA fine-tunes the language layers (the vision encoder stays frozen — it already reads text well)
3. Early stopping per round, then accuracy-gated extra rounds on fresh data until it hits the target
4. Exports **two files** — `seishin-vision-Q4_K_M.gguf` (~1.1 GB) and `mmproj-seishin-vision-Q8_0.gguf`
   (~0.45 GB) — and smoke-tests them with a real image through llama.cpp's server (`--jinja --mmproj`),
   the same code llama.rn runs on the phone

On the phone: Settings → On-device model → pick the model `.gguf`, then **Vision projector** → pick the `mmproj` file.

In [ ]:
# ── Settings ────────────────────────────────────────────────────────────────
BASE_MODEL   = "unsloth/Qwen3-VL-2B-Instruct"
OUT_NAME     = "seishin-vision"
TRAIN_N      = 1200      # image conversations per round (+40% text-only ones)
EVAL_N       = 120
MAX_EPOCHS   = 2         # round 1; early stopping usually ends sooner
TARGET_ACC   = 0.90      # keep training until exact tool-call accuracy reaches this…
MAX_ROUNDS   = 3         # …or this many rounds have run
EVAL_SAMPLES = 120
QUANT        = "Q4_K_M"
SAVE_TO_DRIVE = True

In [ ]:
%%capture
!pip install -q unsloth   # pulls matching torch / transformers / trl / peft versions
# More fonts → more varied synthetic photos (any that are missing are skipped).
!for p in fonts-dejavu-core fonts-freefont-ttf fonts-liberation fonts-roboto fonts-inter fonts-comic-neue fonts-noto-mono; do apt-get -qq install -y $p >/dev/null 2>&1 || true; done

## 1 · Dataset

In [ ]:
%%writefile tools.json
[
  {
    "type": "function",
    "function": {
      "name": "add_event",
      "description": "Add an event to the calendar",
      "parameters": {
        "type": "object",
        "properties": {
          "title": {
            "type": "string",
            "description": "Event title"
          },
          "startDate": {
            "type": "string",
            "description": "Start date/time in ISO format (e.g. 2024-03-15T14:00:00.000Z)"
          },
          "endDate": {
            "type": "string",
            "description": "End date/time in ISO format"
          },
          "description": {
            "type": "string",
            "description": "Optional event description"
          },
          "notes": {
            "type": "string",
            "description": "Optional freeform notes for the event"
          }
        },
        "required": [
          "title",
          "startDate"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "list_events",
      "description": "List all calendar events",
      "parameters": {
        "type": "object",
        "properties": {},
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "list_inbox",
      "description": "List recent notifications/emails/chats in your inbox. Use this to see what notifications you've received.",
      "parameters": {
        "type": "object",
        "properties": {
          "limit": {
            "type": "number",
            "description": "Max items to return (default 20)"
          },
          "unreadOnly": {
            "type": "boolean",
            "description": "Only show unread items"
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "update_event",
      "description": "Edit an existing calendar event. Identify it by id or by its current title (query), then change any of title, start/end time, or description.",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "Current title (or part of it) of the event to edit"
          },
          "id": {
            "type": "string",
            "description": "Exact event id, if known"
          },
          "title": {
            "type": "string",
            "description": "New title"
          },
          "startDate": {
            "type": "string",
            "description": "New start date/time in ISO format"
          },
          "endDate": {
            "type": "string",
            "description": "New end date/time in ISO format"
          },
          "description": {
            "type": "string",
            "description": "New description"
          },
          "notes": {
            "type": "string",
            "description": "New freeform notes for the event"
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "delete_event",
      "description": "Delete/remove a calendar event, identified by id or by its title (query).",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "Title (or part of it) of the event to delete"
          },
          "id": {
            "type": "string",
            "description": "Exact event id, if known"
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "add_todo",
      "description": "Add a task with an optional due date, a checklist of items to do/answer, and reminders. Use reminder \"persistent\" when the user wants to be reminded until it's done (e.g. homework, bills).",
      "parameters": {
        "type": "object",
        "properties": {
          "title": {
            "type": "string",
            "description": "Task title"
          },
          "priority": {
            "type": "string",
            "description": "Priority: low, medium, or high"
          },
          "dueDate": {
            "type": "string",
            "description": "Optional due date/time in ISO format (local time)"
          },
          "reminder": {
            "type": "string",
            "description": "persistent (keep reminding until done), normal (before the deadline), or off"
          },
          "items": {
            "type": "array",
            "items": {
              "type": "string"
            },
            "description": "Optional checklist: questions or steps, as strings"
          }
        },
        "required": [
          "title"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "list_todos",
      "description": "List tasks, soonest deadline first, with overdue ones flagged, checklist progress and where they came from (Todoist, Notion, a course feed). By default only open tasks.",
      "parameters": {
        "type": "object",
        "properties": {
          "includeCompleted": {
            "type": "boolean",
            "description": "Include completed tasks"
          },
          "withinDays": {
            "type": "number",
            "description": "Only tasks due within this many days (overdue always included)"
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "complete_todo",
      "description": "Mark a todo as done (or reopen it), identified by id or by its title (query).",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "Title (or part of it) of the todo"
          },
          "id": {
            "type": "string",
            "description": "Exact todo id, if known"
          },
          "done": {
            "type": "boolean",
            "description": "true to mark done (default), false to reopen it"
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "update_todo",
      "description": "Edit an existing todo. Identify it by id or by its current title (query), then change title, priority, or due date.",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "Current title (or part of it) of the todo to edit"
          },
          "id": {
            "type": "string",
            "description": "Exact todo id, if known"
          },
          "title": {
            "type": "string",
            "description": "New title"
          },
          "priority": {
            "type": "string",
            "description": "New priority: low, medium, or high"
          },
          "dueDate": {
            "type": "string",
            "description": "New due date in ISO format"
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "delete_todo",
      "description": "Delete/remove a todo, identified by id or by its title (query).",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "Title (or part of it) of the todo to delete"
          },
          "id": {
            "type": "string",
            "description": "Exact todo id, if known"
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "add_note",
      "description": "Create a note in the note-taking app. Use for anything the user wants to jot down, save, or remember as free text (ideas, lists, meeting notes, thoughts) — as opposed to a dated task (add_todo) or a scheduled event (add_event).",
      "parameters": {
        "type": "object",
        "properties": {
          "title": {
            "type": "string",
            "description": "Short note title"
          },
          "body": {
            "type": "string",
            "description": "The note content / body text"
          },
          "tags": {
            "type": "string",
            "description": "Comma-separated tags, e.g. 'work,ideas'"
          },
          "pinned": {
            "type": "boolean",
            "description": "Pin the note to the top"
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "list_notes",
      "description": "List saved notes (output includes each note's id). Optionally filter by a search query matching title, body, or tags.",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "Optional text to filter notes by"
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "update_note",
      "description": "Edit an existing note. Identify it by id or by its current title (query), then change title, body, tags, or pin.",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "Current title (or part of it) of the note to edit"
          },
          "id": {
            "type": "string",
            "description": "Exact note id, if known"
          },
          "title": {
            "type": "string",
            "description": "New title"
          },
          "body": {
            "type": "string",
            "description": "New body text (replaces the existing body)"
          },
          "tags": {
            "type": "string",
            "description": "New comma-separated tags (replaces existing tags)"
          },
          "pinned": {
            "type": "boolean",
            "description": "Pin (true) or unpin (false) the note"
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "remember_entity",
      "description": "Store an entity in your knowledge graph. Use this when the user tells you something important about a person, place, concept, or thing that they might want you to recall later.",
      "parameters": {
        "type": "object",
        "properties": {
          "name": {
            "type": "string",
            "description": "Entity name"
          },
          "type": {
            "type": "string",
            "description": "Entity type (person, place, concept, project, preference, etc.)"
          },
          "description": {
            "type": "string",
            "description": "What to remember about this entity"
          },
          "tags": {
            "type": "string",
            "description": "Optional comma-separated tags"
          }
        },
        "required": [
          "name",
          "type",
          "description"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "recall_memory",
      "description": "Search your knowledge graph. Call this when the user asks about something they told you earlier, or when you need context from prior conversations.",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "What to search for in the knowledge graph"
          }
        },
        "required": [
          "query"
        ]
      }
    }
  }
]

In [ ]:
%%writefile generate_dataset.py
#!/usr/bin/env python3
"""
Synthetic tool-calling dataset for Seishin's on-device agent.

Every example is a full OpenAI-style chat — the exact system prompt that
`src/services/local-agent.ts` builds, the user turn, the assistant's tool calls,
the tool results in the exact strings the app's tools return, and the short
final reply. Tool schemas come from `tools.json`, dumped from the app's own
`createDefaultTools()` + `LOCAL_TOOL_NAMES`, so the model learns the real API.

    python3 generate_dataset.py --train 3000 --eval 300 --out data/

Deterministic for a given --seed. No dependencies beyond the stdlib.
"""
from __future__ import annotations

import argparse
import json
import random
import string
from dataclasses import dataclass
from datetime import datetime, timedelta
from pathlib import Path

HERE = Path(__file__).resolve().parent
WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August",
          "September", "October", "November", "December"]

# --------------------------------------------------------------------------- #
# Formatting that mirrors what the app (Hermes, en-US) prints                  #
# --------------------------------------------------------------------------- #

def hm12(d: datetime) -> str:
    """toLocaleTimeString en-US 2-digit: '09:05 AM'."""
    return d.strftime("%I:%M %p")


def system_prompt(now: datetime, known: str = "") -> str:
    """Byte-for-byte the prompt from createLocalSystemPrompt() in local-agent.ts."""
    date = f"{WEEKDAYS[now.weekday()]}, {MONTHS[now.month - 1]} {now.day}, {now.year}"
    context = f"\n\nWhat you know:\n{known[:1500]}" if known.strip() else ""
    return f"""You are Seishin, an assistant running fully on the user's phone, offline.
Today is {date}, {hm12(now)}.
Use the tools to add, read, change or delete the user's events, tasks and notes, and to remember facts about them.
- When the user asks for an action, call the tool right away. Never ask permission.
- "task", "todo", "homework", "deadline" → add_todo. "meeting", "event", "at <time>" → add_event. "note", "write down" → add_note.
- Resolve "tomorrow", "next Friday" etc. from today's date.
- After tools run, reply in one or two short sentences. If no tool fits, just answer.{context}"""


def iso(d: datetime) -> str:
    """Local ISO datetime, no zone — `new Date()` reads it as phone-local time."""
    return d.strftime("%Y-%m-%dT%H:%M:%S")


def locale_string(d: datetime) -> str:
    """new Date().toLocaleString(): '11/12/2026, 3:00:00 PM'."""
    return f"{d.month}/{d.day}/{d.year}, {d.hour % 12 or 12}:{d.minute:02d}:00 {'AM' if d.hour < 12 else 'PM'}"


def short_due(d: datetime) -> str:
    """toLocaleString([], {weekday:'short', month:'short', day, hour, minute}): 'Thu, Nov 12, 3:00 PM'."""
    return f"{WEEKDAYS[d.weekday()][:3]}, {MONTHS[d.month - 1][:3]} {d.day}, {d.hour % 12 or 12}:{d.minute:02d} {'AM' if d.hour < 12 else 'PM'}"


def human_time(d: datetime) -> str:
    return f"{d.hour % 12 or 12}:{d.minute:02d} {'AM' if d.hour < 12 else 'PM'}"


def human_day(d: datetime, now: datetime) -> str:
    delta = (d.date() - now.date()).days
    if delta == 0:
        return "today"
    if delta == 1:
        return "tomorrow"
    return f"{WEEKDAYS[d.weekday()]}, {MONTHS[d.month - 1][:3]} {d.day}"


def app_id(rng: random.Random, prefix: str) -> str:
    """Looks like uid(prefix) from utils/id.ts."""
    t = "".join(rng.choice(string.digits + "abcdefghijklmnopqrstuvwxyz") for _ in range(8))
    r = "".join(rng.choice(string.digits + "abcdefghijklmnopqrstuvwxyz") for _ in range(4))
    return f"{prefix}-{t}-{rng.randint(1, 999):x}-{r}"


# --------------------------------------------------------------------------- #
# Vocabulary                                                                   #
# --------------------------------------------------------------------------- #

PEOPLE = ["Mia", "Josh", "Ana", "Carlo", "Liza", "Ben", "Aiko", "Sam", "Nico", "Jade", "Paolo",
          "Rhea", "Marco", "Kai", "Elena", "Tomas", "Bea", "Leo", "Grace", "Dr. Santos", "Prof. Reyes"]
EVENTS = ["Dentist appointment", "Team standup", "Project sync", "Gym session", "Doctor's checkup",
          "Job interview", "Client call", "Yoga class", "Haircut", "Piano lesson", "Group study",
          "Thesis defense", "Sprint planning", "Car service", "Board meeting", "Parent-teacher meeting",
          "Movie night", "Basketball practice", "Church service", "Flight to Cebu", "Code review",
          "Coffee chat", "Design review", "1:1 with manager", "Vet appointment", "Birthday party",
          "Swimming lesson", "Lab session", "Seminar", "Concert"]
WITH_PERSON = ["Lunch with {p}", "Dinner with {p}", "Call with {p}", "Meeting with {p}",
               "Coffee with {p}", "Study session with {p}", "Date with {p}", "Catch-up with {p}"]
TASKS = ["Math homework", "Pay electricity bill", "Submit lab report", "Renew passport",
         "Buy groceries", "Finish slides", "Email the landlord", "Read chapter 5", "Fix the bike",
         "Pay rent", "Physics problem set", "Update resume", "Book flights", "Clean the room",
         "Water the plants", "Call the bank", "Return library books", "Write essay draft",
         "Review pull request", "Pay internet bill", "Prepare presentation", "History reading",
         "Submit tax forms", "Order birthday gift", "Laundry", "Study for chemistry quiz",
         "Reply to Prof. Reyes", "Backup photos", "Cancel gym membership", "Finish capstone chapter 2"]
CHECKLISTS = {
    "Buy groceries": ["Eggs", "Milk", "Rice", "Bread", "Coffee", "Bananas", "Chicken", "Soap"],
    "Physics problem set": ["Question 1", "Question 2", "Question 3", "Question 4", "Question 5"],
    "Prepare presentation": ["Outline", "Slides", "Speaker notes", "Rehearse"],
    "Write essay draft": ["Outline", "Introduction", "Body", "Conclusion", "References"],
    "Math homework": ["Exercise 1", "Exercise 2", "Exercise 3", "Exercise 4"],
    "Book flights": ["Compare prices", "Pick seats", "Pay"],
}
NOTE_TOPICS = [
    ("App ideas", "A habit tracker that works offline; a shared grocery list over P2P", "ideas"),
    ("Wifi password", "Home: sunflower-2026, Office: ask IT", "home"),
    ("Book recommendations", "Project Hail Mary, Piranesi, The Midnight Library", "books"),
    ("Meeting notes", "Ship v2 by Friday. Josh owns QA, Mia owns release notes.", "work"),
    ("Gift ideas for Mom", "Orchid plant, scarf, cooking class voucher", "family,gifts"),
    ("Workout plan", "Mon: legs, Wed: push, Fri: pull, Sat: run 5k", "fitness"),
    ("Recipe: adobo", "Chicken, soy sauce, vinegar, garlic, bay leaves, pepper. Simmer 40 min.", "recipes"),
    ("Thesis outline", "1. Intro 2. Related work 3. Method 4. Results 5. Discussion", "school"),
    ("Packing list", "Passport, charger, toothbrush, 3 shirts, jacket", "travel"),
    ("Quotes", "Simplicity is prerequisite for reliability.", "quotes"),
    ("Car parking", "Level B2, slot 47 near the elevator", ""),
    ("Movie list", "Dune Part Two, Past Lives, Spirited Away", "movies"),
    ("Budget", "Rent 12000, food 6000, transport 2000, savings 5000", "money"),
    ("Startup notes", "Talk to 10 users before building anything", "ideas,work"),
]
FACTS = [  # (user says, name, type, description)
    ("My sister's name is Ana and she lives in Davao", "Ana", "person", "User's sister; lives in Davao"),
    ("I'm allergic to peanuts", "Peanut allergy", "preference", "User is allergic to peanuts"),
    ("My manager is Josh", "Josh", "person", "User's manager"),
    ("I work at Globe as a data analyst", "Globe", "organization", "User works there as a data analyst"),
    ("I study computer science at UP Diliman", "UP Diliman", "place", "User studies computer science there"),
    ("my birthday is March 14", "User birthday", "event", "User's birthday is March 14"),
    ("I prefer meetings in the morning", "Morning meetings", "preference", "User prefers meetings in the morning"),
    ("My dog's name is Mochi", "Mochi", "pet", "User's dog"),
    ("I'm vegetarian", "Vegetarian", "preference", "User is vegetarian"),
    ("Carlo is my best friend from high school", "Carlo", "person", "User's best friend from high school"),
    ("My capstone project is called Seishin", "Seishin", "project", "User's capstone project"),
    ("I go to the gym at Anytime Fitness in BGC", "Anytime Fitness BGC", "place", "User's gym"),
    ("my girlfriend is Liza", "Liza", "person", "User's girlfriend"),
    ("I usually sleep at 11pm", "Sleep schedule", "preference", "User usually sleeps at 11pm"),
]
SMALL_TALK = [
    ("hi", "Hi! What can I help you with today?"),
    ("hello", "Hello! Need to add something to your calendar or tasks?"),
    ("thanks!", "You're welcome!"),
    ("thank you", "Anytime."),
    ("who are you?", "I'm Seishin, your assistant. I run fully on your phone and can manage your events, tasks and notes."),
    ("what can you do?", "I can add, find, change or delete your events, tasks and notes, check your inbox, and remember things you tell me."),
    ("are you online?", "No — I run entirely on your phone, offline. Your data stays on the device."),
    ("good morning", "Good morning! Want a look at what's on today?"),
    ("tell me a joke", "Why did the calendar break up with the clock? It needed more dates."),
    ("how do I stay focused while studying?", "Try 25-minute focus blocks with 5-minute breaks, and keep your phone out of reach. I can add study blocks to your calendar if you like."),
    ("what's 15% of 2400?", "15% of 2400 is 360."),
    ("convert 5 km to miles", "5 km is about 3.11 miles."),
    ("ok", "Okay! Let me know if you need anything."),
    ("give me a tip for better sleep", "Keep a fixed bedtime and skip screens for the last 30 minutes before sleeping."),
    ("what's the capital of Japan?", "Tokyo."),
    ("is it better to study at night or morning?", "Mornings work best for most people since you're rested, but consistency matters more than the hour."),
    ("salamat", "Walang anuman!"),
    ("kumusta?", "Ayos lang! Ano'ng maitutulong ko?"),
    ("how many days are in a leap year?", "366."),
    ("what's a good way to plan my week?", "List your fixed commitments first, then block time for your top three priorities. I can put those blocks on your calendar."),
    ("can you explain what a habit stack is?", "It's linking a new habit to one you already do, like stretching right after brushing your teeth."),
    ("what should I eat before a workout?", "Something light with carbs and a little protein about an hour before, like a banana with yogurt."),
    ("how long should a power nap be?", "About 10 to 20 minutes, so you wake up before deep sleep."),
    ("what does ASAP mean?", "As soon as possible."),
    ("translate 'good night' to Japanese", "Oyasumi nasai (おやすみなさい)."),
    ("what's 12 times 17?", "204."),
    ("how do I say no to extra work politely?", "Thank them, say your plate is full right now, and offer a later time or an alternative."),
    ("you're helpful", "Thanks! Happy to help."),
    ("never mind", "No problem."),
    ("what's the pomodoro technique?", "Work for 25 minutes, take a 5-minute break, and after four rounds take a longer 15–30 minute break."),
    ("do you store my data in the cloud?", "No. Everything stays on your phone."),
    ("bye", "Bye! Talk soon."),
    ("how far is the moon?", "About 384,000 km from Earth on average."),
    ("give me a motivational quote", "\"It always seems impossible until it's done.\" — Nelson Mandela"),
]
INBOX = [
    ("gmail", "Prof. Reyes", "Quiz moved to Thursday 10 AM"),
    ("messenger", "Mia", "Are we still on for lunch tomorrow?"),
    ("gmail", "Meralco", "Your bill of ₱2,340 is due Nov 20"),
    ("sms", "GCash", "You received ₱500 from Carlo"),
    ("slack", "Josh", "Can you review my PR before 5?"),
    ("gmail", "Shopee", "Your order has shipped"),
    ("viber", "Mom", "Call me when you're free"),
    ("gmail", "HR", "Interview scheduled Monday 2 PM"),
]

# --------------------------------------------------------------------------- #
# Time expressions: (phrase, resolved datetime)                                #
# --------------------------------------------------------------------------- #

def next_weekday(now: datetime, wd: int) -> datetime:
    days = (wd - now.weekday()) % 7 or 7
    return now + timedelta(days=days)


def pick_day(rng: random.Random, now: datetime) -> tuple[str, datetime]:
    r = rng.random()
    if r < 0.18:
        return "today", now
    if r < 0.45:
        return rng.choice(["tomorrow", "tmrw", "tomorrow", "bukas"]), now + timedelta(days=1)
    if r < 0.52:
        return "the day after tomorrow", now + timedelta(days=2)
    if r < 0.75:
        wd = rng.randrange(7)
        while wd == now.weekday():
            wd = rng.randrange(7)
        d = next_weekday(now, wd)
        return rng.choice([f"on {WEEKDAYS[wd]}", WEEKDAYS[wd].lower(), f"this {WEEKDAYS[wd]}", f"next {WEEKDAYS[wd]}"]), d
    if r < 0.85:
        n = rng.randint(2, 10)
        return f"in {n} days", now + timedelta(days=n)
    d = now + timedelta(days=rng.randint(3, 60))
    return rng.choice([f"on {MONTHS[d.month - 1]} {d.day}", f"on {MONTHS[d.month - 1][:3]} {d.day}", f"{d.month}/{d.day}"]), d


def pick_time(rng: random.Random) -> tuple[str, int, int]:
    r = rng.random()
    if r < 0.12:
        return "at noon", 12, 0
    if r < 0.2:
        return rng.choice([("in the morning", 9, 0), ("tonight", 20, 0), ("in the evening", 19, 0), ("in the afternoon", 15, 0)])
    h = rng.choice([7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20])
    m = rng.choice([0, 0, 0, 0, 15, 30, 30, 45])
    h12, ap = h % 12 or 12, "am" if h < 12 else "pm"
    if m == 0:
        txt = rng.choice([f"at {h12}{ap}", f"at {h12} {ap}", f"at {h12}{ap.upper()}", f"{h12}{ap}", f"at {h:02d}:00"])
    else:
        txt = rng.choice([f"at {h12}:{m:02d}{ap}", f"at {h12}:{m:02d} {ap}", f"{h12}:{m:02d}{ap}"])
    return txt, h, m


def when(rng: random.Random, now: datetime) -> tuple[str, datetime]:
    dtxt, day = pick_day(rng, now)
    for _ in range(10):
        ttxt, h, m = pick_time(rng)
        dt = day.replace(hour=h, minute=m, second=0, microsecond=0)
        if dt > now:
            break
    else:
        dtxt, day = "tomorrow", now + timedelta(days=1)
        ttxt, h, m = "at 10am", 10, 0
        dt = day.replace(hour=h, minute=m, second=0, microsecond=0)
    phrase = f"{dtxt} {ttxt}" if rng.random() < 0.7 else f"{ttxt} {dtxt}"
    return phrase, dt


# --------------------------------------------------------------------------- #
# Conversation builder                                                         #
# --------------------------------------------------------------------------- #

@dataclass
class Ctx:
    rng: random.Random
    now: datetime
    n_calls: int = 0

    def call(self, name: str, args: dict) -> dict:
        self.n_calls += 1
        return {"id": f"call_{self.n_calls}", "type": "function", "function": {"name": name, "arguments": args}}


def assistant_calls(*calls: dict) -> dict:
    return {"role": "assistant", "content": "", "tool_calls": list(calls)}


def tool_msg(call: dict, content: str) -> dict:
    return {"role": "tool", "tool_call_id": call["id"], "name": call["function"]["name"], "content": content}


def say(text: str) -> dict:
    return {"role": "assistant", "content": text}


def user(text: str, rng: random.Random) -> dict:
    for a, b in (("to at ", "to "), ("to on ", "to "), ("by on ", "by "), ("due on ", "due "), ("it to in ", "it to ")):
        text = text.replace(a, b)
    if rng.random() < 0.25:
        text = text.lower()
    if rng.random() < 0.3:
        text = text.rstrip(".!?")
    elif rng.random() < 0.2 and not text.endswith(("?", "!", ".")):
        text += rng.choice([".", "!", " pls", " please"])
    return {"role": "user", "content": text}


def cap(s: str) -> str:
    return s[:1].upper() + s[1:]


# --- scenarios: each returns a list of messages after the system prompt ----- #

def s_add_event(c: Ctx) -> list[dict]:
    rng = c.rng
    title = rng.choice(EVENTS) if rng.random() < 0.6 else rng.choice(WITH_PERSON).format(p=rng.choice(PEOPLE))
    phrase, dt = when(rng, c.now)
    args = {"title": title, "startDate": iso(dt)}
    dur_txt = ""
    if rng.random() < 0.3:
        hours = rng.choice([0.5, 1.5, 2, 3])
        args["endDate"] = iso(dt + timedelta(hours=hours))
        dur_txt = rng.choice([f" for {hours:g} hours" if hours != 0.5 else " for 30 minutes", f" for {int(hours * 60)} mins"])
    extra = ""
    if rng.random() < 0.15:
        loc = rng.choice(["at the clinic on 5th Ave", "in room 302", "on Zoom", "at SM North", "at the office"])
        args["description"] = cap(loc.removeprefix("at ").removeprefix("in ").removeprefix("on ").removeprefix("the "))
        extra = f" {loc}"
    t = title.lower() if rng.random() < 0.5 else title
    templates = [
        f"Add {t} {phrase}{dur_txt}{extra}",
        f"Schedule {t} {phrase}{dur_txt}{extra}",
        f"I have {t} {phrase}{extra}",
        f"put {t} on my calendar {phrase}{dur_txt}",
        f"Remind me about {t} {phrase}{extra}",
        f"Can you add an event: {t}, {phrase}{dur_txt}",
        f"{cap(t)} {phrase}{extra}",
        f"set a meeting — {t} {phrase}{dur_txt}",
        f"Pa-add naman ng {t} {phrase}",
        f"new event {t} {phrase}{extra}",
    ]
    msg = rng.choice(templates)
    if "endDate" in args and dur_txt not in msg:
        args.pop("endDate")
    if "description" in args and extra not in msg:
        args.pop("description")
    call = c.call("add_event", args)
    reply = rng.choice([
        f"Added \"{title}\" {human_day(dt, c.now)} at {human_time(dt)}.",
        f"Done — {title} is on your calendar for {human_day(dt, c.now)} at {human_time(dt)}.",
        f"Scheduled {title} for {human_day(dt, c.now)}, {human_time(dt)}.",
    ])
    return [user(msg, rng), assistant_calls(call),
            tool_msg(call, f"Added event: \"{title}\" on {locale_string(dt)}"), say(reply)]


def s_add_todo(c: Ctx) -> list[dict]:
    rng = c.rng
    title = rng.choice(TASKS)
    args: dict = {"title": title}
    parts = []
    due = None
    if rng.random() < 0.75:
        dtxt, day = pick_day(rng, c.now)
        if rng.random() < 0.5:
            due = day.replace(hour=23, minute=59, second=0, microsecond=0)
            parts.append(rng.choice([f"by {dtxt}".replace("by on ", "by "), f"due {dtxt}".replace("due on ", "due "), f"{dtxt}"]))
        else:
            ttxt, h, m = pick_time(rng)
            due = day.replace(hour=h, minute=m, second=0, microsecond=0)
            if due <= c.now:
                due = due + timedelta(days=1)
                dtxt = "tomorrow"
            parts.append(f"due {dtxt} {ttxt}".replace("due on ", "due "))
        args["dueDate"] = iso(due)
    priority_txt = ""
    if rng.random() < 0.25:
        p = rng.choice(["high", "high", "low"])
        args["priority"] = p
        priority_txt = rng.choice({"high": ["urgent", "important", "high priority"], "low": ["low priority", "not urgent"]}[p])
    persistent = False
    if rng.random() < 0.25:
        persistent = True
        args["reminder"] = "persistent"
    items = None
    if title in CHECKLISTS and rng.random() < 0.6:
        pool = CHECKLISTS[title]
        items = rng.sample(pool, k=rng.randint(2, min(5, len(pool))))
        args["items"] = items
    word = rng.choice(["task", "todo", "to-do", "reminder"]) if "homework" not in title.lower() else "homework"
    when_txt = (" " + parts[0]) if parts else ""
    pr = f" ({priority_txt})" if priority_txt else ""
    keep = rng.choice([" and keep reminding me until it's done", ", remind me until I finish it", " — nag me until it's done"]) if persistent else ""
    lst = (": " + ", ".join(items)) if items else ""
    t = title.lower() if rng.random() < 0.6 else title
    msg = rng.choice([
        f"Add a {word} to {t}{when_txt}{pr}{keep}{lst}",
        f"I need to {t}{when_txt}{pr}{keep}{lst}",
        f"new {word}: {t}{when_txt}{pr}{keep}{lst}",
        f"remind me to {t}{when_txt}{keep}{pr}{lst}",
        f"{cap(t)}{when_txt}{pr}{keep}{lst}",
        f"todo {t}{when_txt}{pr}{lst}{keep}",
    ]).replace(" to to ", " to ")
    call = c.call("add_todo", args)
    res = f"Added task \"{title}\""
    if due:
        res += f" due {short_due(due)}"
    if items:
        res += f" with {len(items)} checklist item{'s' if len(items) > 1 else ''}"
    if persistent:
        res += ". I'll keep reminding until it's done"
    res += "."
    reply = f"Added \"{title}\"" + (f", due {human_day(due, c.now)} at {human_time(due)}" if due else "") + "."
    if persistent:
        reply += " I'll keep reminding you until it's done."
    elif items:
        reply += f" It has {len(items)} checklist items."
    return [user(msg, rng), assistant_calls(call), tool_msg(call, res), say(reply)]


def s_add_note(c: Ctx) -> list[dict]:
    rng = c.rng
    title, body, tags = rng.choice(NOTE_TOPICS)
    args: dict = {"title": title, "body": body}
    msg = rng.choice([
        f"Write down: {body}",
        f"Note this — {title.lower()}: {body}",
        f"Save a note called {title}: {body}",
        f"take a note: {body}",
        f"jot down my {title.lower()}: {body}",
        f"Make a note \"{title}\" — {body}",
    ])
    if tags and rng.random() < 0.4:
        args["tags"] = tags
        msg += f" (tag it {tags.replace(',', ' and ')})"
    if rng.random() < 0.15:
        args["pinned"] = True
        msg += " and pin it"
    call = c.call("add_note", args)
    return [user(msg, rng), assistant_calls(call), tool_msg(call, f"Added note: \"{title}\""),
            say(rng.choice([f"Saved your note \"{title}\".", f"Noted — \"{title}\" is in your notes.", f"Saved \"{title}\"" + (" and pinned it." if args.get("pinned") else "."), ]))]


def fake_events(c: Ctx, k: int) -> list[tuple[str, str, datetime]]:
    rng = c.rng
    out = []
    for _ in range(k):
        _, dt = when(rng, c.now)
        title = rng.choice(EVENTS) if rng.random() < 0.6 else rng.choice(WITH_PERSON).format(p=rng.choice(PEOPLE))
        out.append((app_id(rng, "evt"), title, dt))
    return sorted(out, key=lambda e: e[2])


def event_lines(evts) -> str:
    return "\n".join(f"- [id:{i}] {t} ({d.month}/{d.day}/{d.year} {hm12(d)})" for i, t, d in evts)


def s_list_events(c: Ctx) -> list[dict]:
    rng = c.rng
    evts = fake_events(c, rng.randint(0, 5))
    msg = rng.choice(["What's on my calendar?", "what do I have coming up", "show my events", "any meetings this week?",
                      "What's my schedule?", "do I have anything tomorrow?", "list my events", "Ano schedule ko?"])
    call = c.call("list_events", {})
    if not evts:
        return [user(msg, rng), assistant_calls(call), tool_msg(call, "No events found."),
                say("Your calendar is clear — no events coming up.")]
    first = evts[0]
    summary = "; ".join(f"{t} {human_day(d, c.now)} at {human_time(d)}" for _, t, d in evts[:3])
    reply = (f"You have {len(evts)} event{'s' if len(evts) > 1 else ''}: {summary}"
             + (f", and {len(evts) - 3} more." if len(evts) > 3 else "."))
    if "tomorrow" in msg:
        tm = [e for e in evts if (e[2].date() - c.now.date()).days == 1]
        reply = ("Tomorrow you have " + "; ".join(f"{t} at {human_time(d)}" for _, t, d in tm) + ".") if tm else \
            f"Nothing tomorrow. Next up is {first[1]} {human_day(first[2], c.now)} at {human_time(first[2])}."
    return [user(msg, rng), assistant_calls(call), tool_msg(call, event_lines(evts)), say(reply)]


def fake_todos(c: Ctx, k: int):
    rng = c.rng
    out = []
    for title in rng.sample(TASKS, k):
        due = c.now + timedelta(days=rng.randint(-2, 9), hours=rng.randint(0, 10))
        due = due.replace(minute=rng.choice([0, 30, 59]), second=0, microsecond=0)
        out.append((app_id(rng, "todo"), title, rng.choice(["medium", "medium", "high", "low"]), due))
    return sorted(out, key=lambda t: t[3])


def s_list_todos(c: Ctx) -> list[dict]:
    rng = c.rng
    todos = fake_todos(c, rng.randint(0, 5))
    args: dict = {}
    msg = rng.choice(["What tasks do I have?", "show my todos", "what's due?", "anything overdue?", "my to-do list",
                      "what homework do I have left", "list my tasks"])
    if rng.random() < 0.3:
        n = rng.choice([1, 3, 7])
        args["withinDays"] = n
        msg = rng.choice({1: ["what's due today or tomorrow?", "anything due by tomorrow?"],
                          3: ["what's due in the next 3 days?", "tasks for the next few days"],
                          7: ["what's due this week?", "tasks due within a week"]}[n])
        todos = [t for t in todos if t[3] <= c.now + timedelta(days=n)]
    call = c.call("list_todos", args)
    if not todos:
        return [user(msg, rng), assistant_calls(call), tool_msg(call, "No tasks found."),
                say("You're all caught up — no open tasks.")]
    lines = []
    for i, t, p, d in todos:
        s = f"- [id:{i}] {t} [{p}] due {short_due(d)}"
        if d < c.now:
            s += " — OVERDUE"
        lines.append(s)
    overdue = [t for t in todos if t[3] < c.now]
    reply = f"You have {len(todos)} open task{'s' if len(todos) > 1 else ''}"
    if overdue:
        reply += f"; {', '.join(t[1] for t in overdue)} {'is' if len(overdue) == 1 else 'are'} overdue"
    nxt = [t for t in todos if t[3] >= c.now]
    if nxt:
        reply += f". Next up: {nxt[0][1]}, due {human_day(nxt[0][3], c.now)} at {human_time(nxt[0][3])}"
    return [user(msg, rng), assistant_calls(call), tool_msg(call, "\n".join(lines)), say(reply + ".")]


def s_complete_todo(c: Ctx) -> list[dict]:
    rng = c.rng
    title = rng.choice(TASKS)
    q = title.lower() if rng.random() < 0.5 else " ".join(title.lower().split()[:2])
    done = rng.random() < 0.85
    if done:
        msg = rng.choice([f"I finished {q}", f"mark {q} as done", f"done with {q}", f"check off {q}", f"tapos na {q}"])
        args = {"query": q}
    else:
        msg = rng.choice([f"reopen {q}", f"mark {q} as not done", f"I'm not actually done with {q}"])
        args = {"query": q, "done": False}
    call = c.call("complete_todo", args)
    return [user(msg, rng), assistant_calls(call),
            tool_msg(call, f"Marked todo \"{title}\" as {'done' if done else 'not done'}."),
            say(rng.choice([f"Nice work — \"{title}\" is done.", f"Checked off \"{title}\"."]) if done else f"Reopened \"{title}\".")]


def s_update_event(c: Ctx) -> list[dict]:
    rng = c.rng
    title = rng.choice(EVENTS)
    q = title.lower() if rng.random() < 0.5 else title.split()[0].lower()
    phrase, dt = when(rng, c.now)
    kind = rng.random()
    if kind < 0.6:
        msg = rng.choice([f"Move my {q} to {phrase}", f"reschedule {q} to {phrase}", f"change {q} to {phrase}",
                          f"push the {q} to {phrase}"])
        args = {"query": q, "startDate": iso(dt)}
        reply = f"Moved \"{title}\" to {human_day(dt, c.now)} at {human_time(dt)}."
        new_title = title
    elif kind < 0.8:
        new_title = rng.choice(EVENTS)
        msg = f"rename {q} to {new_title}"
        args = {"query": q, "title": new_title}
        reply = f"Renamed it to \"{new_title}\"."
    else:
        note = rng.choice(["Bring the documents", "Room changed to 401", "Wear formal attire", "Bring laptop"])
        msg = f"add a note to my {q}: {note.lower()}"
        args = {"query": q, "notes": note}
        reply = f"Added that note to \"{title}\"."
        new_title = title
    call = c.call("update_event", args)
    return [user(msg, rng), assistant_calls(call), tool_msg(call, f"Updated event \"{new_title}\"."), say(reply)]


def s_update_todo(c: Ctx) -> list[dict]:
    rng = c.rng
    title = rng.choice(TASKS)
    q = title.lower()
    if rng.random() < 0.5:
        dtxt, day = pick_day(rng, c.now)
        due = day.replace(hour=23, minute=59, second=0, microsecond=0)
        msg = rng.choice([f"push {q} to {dtxt}", f"change the deadline of {q} to {dtxt}", f"{q} is now due {dtxt}"]).replace("to on ", "to ").replace("due on ", "due ")
        args = {"query": q, "dueDate": iso(due)}
        reply = f"\"{title}\" is now due {human_day(due, c.now)}."
    else:
        p = rng.choice(["high", "low"])
        msg = rng.choice([f"make {q} {p} priority", f"set {q} priority to {p}", f"{q} is {'urgent' if p == 'high' else 'not urgent'}"])
        args = {"query": q, "priority": p}
        reply = f"Set \"{title}\" to {p} priority."
    call = c.call("update_todo", args)
    return [user(msg, rng), assistant_calls(call), tool_msg(call, f"Updated todo \"{title}\"."), say(reply)]


def s_delete(c: Ctx) -> list[dict]:
    rng = c.rng
    kind = rng.choice(["event", "todo"])
    title = rng.choice(EVENTS if kind == "event" else TASKS)
    q = title.lower()
    msg = rng.choice([f"delete {q}", f"cancel my {q}", f"remove {q} from my {'calendar' if kind == 'event' else 'tasks'}",
                      f"I don't need {q} anymore, delete it", f"pakitanggal {q}"])
    call = c.call(f"delete_{kind}", {"query": q})
    return [user(msg, rng), assistant_calls(call), tool_msg(call, f"Deleted {kind} \"{title}\"."),
            say(rng.choice([f"Deleted \"{title}\".", f"Removed \"{title}\"."]))]


def s_ambiguous(c: Ctx) -> list[dict]:
    """Tool says several items match → the model asks, it does not guess."""
    rng = c.rng
    p1, p2 = rng.sample(PEOPLE, 2)
    msg = rng.choice(["delete my lunch", "cancel lunch", "move lunch to 2pm"])
    tool = "update_event" if "move" in msg else "delete_event"
    args = {"query": "lunch"}
    if tool == "update_event":
        _, dt = when(rng, c.now)
        dt = dt.replace(hour=14, minute=0)
        args["startDate"] = iso(dt)
    call = c.call(tool, args)
    return [user(msg, rng), assistant_calls(call),
            tool_msg(call, f"Error: Multiple events match \"lunch\": Lunch with {p1}, Lunch with {p2}. Ask which one or pass its id."),
            say(f"You have two lunches — with {p1} and with {p2}. Which one?")]


def s_list_then_act(c: Ctx) -> list[dict]:
    """Multi-step: read first, then act on the id found."""
    rng = c.rng
    evts = fake_events(c, rng.randint(2, 4))
    target = rng.choice(evts)
    msg = rng.choice(["cancel my next meeting", "delete the first event on my calendar", "move my next event an hour later"])
    if "next" in msg or "first" in msg:
        target = evts[0]
    c1 = c.call("list_events", {})
    out = [user(msg, rng), assistant_calls(c1), tool_msg(c1, event_lines(evts))]
    if "move" in msg:
        nd = target[2] + timedelta(hours=1)
        c2 = c.call("update_event", {"id": target[0], "startDate": iso(nd)})
        out += [assistant_calls(c2), tool_msg(c2, f"Updated event \"{target[1]}\"."),
                say(f"Moved \"{target[1]}\" to {human_time(nd)} {human_day(nd, c.now)}.")]
    else:
        c2 = c.call("delete_event", {"id": target[0]})
        out += [assistant_calls(c2), tool_msg(c2, f"Deleted event \"{target[1]}\"."),
                say(f"Cancelled \"{target[1]}\" ({human_day(target[2], c.now)} at {human_time(target[2])}).")]
    return out


def s_multi(c: Ctx) -> list[dict]:
    """Two actions in one message → two parallel tool calls."""
    rng = c.rng
    title = rng.choice(EVENTS)
    phrase, dt = when(rng, c.now)
    task = rng.choice(TASKS)
    dtxt, day = pick_day(rng, c.now)
    due = day.replace(hour=23, minute=59, second=0, microsecond=0)
    msg = rng.choice([f"add {title.lower()} {phrase} and a task to {task.lower()} by {dtxt}",
                      f"Schedule {title.lower()} {phrase}, also remind me to {task.lower()} by {dtxt}"]).replace("by on ", "by ")
    c1 = c.call("add_event", {"title": title, "startDate": iso(dt)})
    c2 = c.call("add_todo", {"title": task, "dueDate": iso(due)})
    return [user(msg, rng), assistant_calls(c1, c2),
            tool_msg(c1, f"Added event: \"{title}\" on {locale_string(dt)}"),
            tool_msg(c2, f"Added task \"{task}\" due {short_due(due)}."),
            say(f"Added {title} {human_day(dt, c.now)} at {human_time(dt)}, and \"{task}\" due {human_day(due, c.now)}.")]


def s_remember(c: Ctx) -> list[dict]:
    rng = c.rng
    said, name, typ, desc = rng.choice(FACTS)
    msg = rng.choice([said, f"remember that {said[0].lower() + said[1:]}", f"fyi {said[0].lower() + said[1:]}"])
    call = c.call("remember_entity", {"name": name, "type": typ, "description": desc})
    return [user(msg, rng), assistant_calls(call), tool_msg(call, f"Remembered \"{name}\" as a {typ}."),
            say(rng.choice(["Got it, I'll remember that.", "Noted — I'll remember.", f"I'll remember {name}."]))]


def s_recall(c: Ctx) -> list[dict]:
    rng = c.rng
    said, name, typ, desc = rng.choice(FACTS)
    qs = {"person": [f"who is {name}?", f"what do you know about {name}"],
          "pet": ["what's my dog's name?"], "place": [f"what did I tell you about {name}?"]}
    msg = rng.choice(qs.get(typ, [f"what do you remember about {name.lower()}?", f"what did I say about {name.lower()}?"]))
    call = c.call("recall_memory", {"query": name})
    hit = rng.random() < 0.85
    res = f"- **{name}** [id:{app_id(rng, 'ent')}] ({typ}): {desc}" if hit else f"No matches for \"{name}\"."
    reply = f"{desc.replace('User', 'You').replace(' user', ' you')}." if hit else f"I don't have anything saved about {name} yet. Want to tell me?"
    return [user(msg, rng), assistant_calls(call), tool_msg(call, res), say(reply.replace("..", "."))]


def s_inbox(c: Ctx) -> list[dict]:
    rng = c.rng
    items = rng.sample(INBOX, rng.randint(1, 4))
    unread = rng.random() < 0.5
    args = {"unreadOnly": True} if unread else {}
    msg = rng.choice(["any new messages?", "check my unread notifications"] if unread else
                     ["what's in my inbox?", "show my notifications", "did anyone message me?"])
    call = c.call("list_inbox", args)
    d = c.now - timedelta(days=rng.randint(0, 2))
    lines = "\n".join(f"- [{s}] {t}: {b} ({d.month}/{d.day}/{d.year}) [unread]" for s, t, b in items)
    reply = f"You have {len(items)} {'unread ' if unread else ''}item{'s' if len(items) > 1 else ''}: " + \
        "; ".join(f"{t} — {b}" for _, t, b in items[:3]) + "."
    return [user(msg, rng), assistant_calls(call), tool_msg(call, lines), say(reply)]


def s_list_notes(c: Ctx) -> list[dict]:
    rng = c.rng
    title, body, tags = rng.choice(NOTE_TOPICS)
    key = title.split()[0].lower()
    msg = rng.choice([f"find my note about {key}", f"what did I write about {key}?", f"show my {title.lower()} note"])
    call = c.call("list_notes", {"query": key})
    nid = app_id(rng, "note")
    tg = f" #{' #'.join(tags.split(','))}" if tags else ""
    return [user(msg, rng), assistant_calls(call), tool_msg(call, f"- [id:{nid}] {title} — {body[:60]}{tg}"),
            say(f"Here it is — \"{title}\": {body}")]


def s_update_note(c: Ctx) -> list[dict]:
    rng = c.rng
    title, body, _ = rng.choice(NOTE_TOPICS)
    if rng.random() < 0.5:
        msg, args, reply = f"pin my {title.lower()} note", {"query": title.lower(), "pinned": True}, f"Pinned \"{title}\"."
    else:
        new = rng.choice(["Updated: ", "v2: "]) + body
        msg = f"replace my {title.lower()} note with: {new}"
        args, reply = {"query": title.lower(), "body": new}, f"Updated \"{title}\"."
    call = c.call("update_note", args)
    return [user(msg, rng), assistant_calls(call), tool_msg(call, f"Updated note \"{title}\"."), say(reply)]


def s_tool_error(c: Ctx) -> list[dict]:
    rng = c.rng
    title = rng.choice(TASKS)
    q = title.lower()
    call = c.call("complete_todo", {"query": q})
    return [user(f"mark {q} done", rng), assistant_calls(call),
            tool_msg(call, f"Error: No todo matches \"{q}\"."),
            say(f"I couldn't find a task called \"{title}\". Want me to add it?")]


def s_chat(c: Ctx) -> list[dict]:
    rng = c.rng
    q, a = rng.choice(SMALL_TALK)
    return [user(q, rng), say(a)]


def s_date_question(c: Ctx) -> list[dict]:
    rng = c.rng
    if rng.random() < 0.5:
        return [user(rng.choice(["what day is it today?", "what's the date?", "anong araw ngayon?"]), rng),
                say(f"Today is {WEEKDAYS[c.now.weekday()]}, {MONTHS[c.now.month - 1]} {c.now.day}, {c.now.year}.")]
    wd = rng.randrange(7)
    d = next_weekday(c.now, wd)
    return [user(f"what's the date this coming {WEEKDAYS[wd]}?", rng),
            say(f"This coming {WEEKDAYS[wd]} is {MONTHS[d.month - 1]} {d.day}.")]


def s_followup(c: Ctx) -> list[dict]:
    """Multi-turn: an earlier exchange in history, then a follow-up that refers back."""
    first = s_add_event(c)
    title = first[1]["tool_calls"][0]["function"]["arguments"]["title"]
    rng = c.rng
    phrase, dt = when(rng, c.now)
    msg = rng.choice([f"actually make it {phrase}", f"move it to {phrase}", f"wait, change that to {phrase}"])
    call = c.call("update_event", {"query": title, "startDate": iso(dt)})
    return first + [user(msg, rng), assistant_calls(call), tool_msg(call, f"Updated event \"{title}\"."),
                    say(f"Moved \"{title}\" to {human_day(dt, c.now)} at {human_time(dt)}.")]


SCENARIOS = [  # (weight, fn)
    (16, s_add_event), (16, s_add_todo), (8, s_add_note), (7, s_list_events), (7, s_list_todos),
    (5, s_complete_todo), (6, s_update_event), (4, s_update_todo), (5, s_delete), (2, s_ambiguous),
    (4, s_list_then_act), (4, s_multi), (4, s_remember), (3, s_recall), (3, s_inbox), (3, s_list_notes),
    (2, s_update_note), (2, s_tool_error), (7, s_chat), (2, s_date_question), (4, s_followup),
]


def known_context(rng: random.Random) -> str:
    """Sometimes the app prepends retrieved memory to the prompt."""
    if rng.random() > 0.25:
        return ""
    facts = rng.sample(FACTS, rng.randint(1, 3))
    return "\n".join(f"- {n} ({t}): {d}" for _, n, t, d in facts)


def make_example(rng: random.Random, tools: list[dict]) -> dict:
    now = datetime(2026, 1, 1) + timedelta(days=rng.randint(0, 729), hours=rng.randint(6, 22), minutes=rng.randint(0, 59))
    c = Ctx(rng, now)
    fn = rng.choices([f for _, f in SCENARIOS], weights=[w for w, _ in SCENARIOS])[0]
    msgs = fn(c)
    return {"scenario": fn.__name__.removeprefix("s_"),
            "messages": [{"role": "system", "content": system_prompt(now, known_context(rng))}] + msgs,
            "tools": tools}


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--train", type=int, default=3000)
    ap.add_argument("--eval", type=int, default=300)
    ap.add_argument("--seed", type=int, default=7)
    ap.add_argument("--tools", default=str(HERE / "tools.json"))
    ap.add_argument("--out", default=str(HERE / "data"))
    a = ap.parse_args()
    tools = json.loads(Path(a.tools).read_text())
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    seen: set[str] = set()
    for split, n, seed in (("train", a.train, a.seed), ("eval", a.eval, a.seed + 1000)):
        rng = random.Random(seed)
        rows = []
        while len(rows) < n:
            ex = make_example(rng, tools)
            key = json.dumps(ex["messages"][1:], sort_keys=True)
            if key in seen:  # keep eval disjoint from train
                continue
            seen.add(key)
            rows.append(ex)
        with open(out / f"{split}.jsonl", "w") as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"{split}: {len(rows)} examples → {out / f'{split}.jsonl'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile generate_vision_dataset.py
#!/usr/bin/env python3
"""
Image + tool-calling dataset for Seishin's multimodal on-device agent.

Renders synthetic photos/screenshots the user would attach in the agent chat —
class schedules, event posters, bills, assignment sheets, chat screenshots,
handwritten notes, plus plain shape pictures for visual grounding — each with
its ground truth, and writes conversations where the model reads the image and
calls the right Seishin tools (add_event / add_todo / add_note), or just answers.

Mixed with text-only conversations from generate_dataset.py so the model keeps
its text tool-calling skills.

    pip install pillow
    python3 generate_vision_dataset.py --train 1600 --eval 160 --out data_vision/

Images are sized to ≤ 262k pixels (≤ 256 Qwen3-VL image tokens), matching the
app's `image_max_tokens: 256`. User turns put the image first, then text —
the order llama.rn renders an attachment in.
"""
from __future__ import annotations

import argparse
import glob
import json
import math
import random
from datetime import datetime, timedelta
from pathlib import Path

from PIL import Image, ImageDraw, ImageFilter, ImageFont

import generate_dataset as G

HERE = Path(__file__).resolve().parent
SIZES = [(512, 512), (448, 576), (576, 448), (384, 640), (640, 384)]

# --------------------------------------------------------------------------- #
# Rendering helpers                                                            #
# --------------------------------------------------------------------------- #

_FONTS: dict[str, list[str]] = {}


def renders_latin(path: str) -> bool:
    """Reject fonts that draw tofu boxes for Latin text (CJK/symbol-only fonts)."""
    try:
        f = ImageFont.truetype(path, 24)
        a, b = f.getmask("abcdefgh"), f.getmask("hgfedcba")
        return a.getbbox() is not None and bytes(a) != bytes(b) and bytes(f.getmask("0123")) != bytes(f.getmask("3210"))
    except OSError:
        return False


def fonts(kind: str) -> list[str]:
    if not _FONTS:
        every = [f for f in glob.glob("/usr/share/fonts/**/*.[ot]tf", recursive=True)
                 if not any(x in f.lower() for x in ("emoji", "symbol", "math", "music", "dingbat", "thin", "light", "hairline"))
                 and renders_latin(f)]
        _FONTS["bold"] = [f for f in every if "bold" in f.lower()] or every
        _FONTS["hand"] = [f for f in every if any(x in f.lower() for x in ("italic", "oblique", "script", "comic"))] or every
        _FONTS["any"] = every
    return _FONTS[kind]


def font(rng: random.Random, size: int, kind: str = "any") -> ImageFont.FreeTypeFont:
    choices = fonts(kind)
    if choices:
        try:
            return ImageFont.truetype(rng.choice(choices), size)
        except OSError:
            pass
    return ImageFont.load_default(size=size)


def fit(d: ImageDraw.ImageDraw, rng: random.Random, text: str, size: int, max_w: int, kind: str = "any"):
    """A font of at most `size` px whose rendering of `text` fits in `max_w` — ground truth must stay visible."""
    path = rng.choice(fonts(kind)) if fonts(kind) else None
    while True:
        f = ImageFont.truetype(path, size) if path else ImageFont.load_default(size=size)
        if d.textlength(text, font=f) <= max_w or size <= 10:
            return f
        size -= 2


def palette(rng: random.Random) -> tuple[tuple, tuple, tuple]:
    """(background, ink, accent) with readable contrast."""
    light = [(255, 255, 255), (250, 247, 240), (240, 244, 248), (255, 252, 230), (236, 240, 233), (245, 245, 245)]
    dark = [(20, 20, 24), (30, 36, 52), (44, 30, 60), (12, 50, 45)]
    accents = [(220, 60, 60), (40, 110, 220), (230, 140, 30), (40, 160, 90), (150, 70, 200), (0, 0, 0)]
    if rng.random() < 0.75:
        return rng.choice(light), rng.choice([(15, 15, 15), (40, 40, 60), (30, 30, 30)]), rng.choice(accents)
    return rng.choice(dark), (245, 245, 245), rng.choice(accents[:-1])


def wrap(draw: ImageDraw.ImageDraw, text: str, f, width: int) -> list[str]:
    lines, cur = [], ""
    for word in text.split():
        nxt = f"{cur} {word}".strip()
        if draw.textlength(nxt, font=f) <= width:
            cur = nxt
        else:
            if cur:
                lines.append(cur)
            cur = word
    return lines + ([cur] if cur else [])


def photo_effects(img: Image.Image, rng: random.Random) -> Image.Image:
    """Make renders look like phone photos: tilt, blur, noise, JPEG-ish softness."""
    if rng.random() < 0.5:
        bg = tuple(rng.randint(60, 200) for _ in range(3))
        img = img.rotate(rng.uniform(-6, 6), resample=Image.BICUBIC, expand=False, fillcolor=bg)
    if rng.random() < 0.35:
        img = img.filter(ImageFilter.GaussianBlur(rng.uniform(0.3, 1.0)))
    if rng.random() < 0.35:
        noise = Image.effect_noise(img.size, rng.uniform(8, 25)).convert("RGB")
        img = Image.blend(img, noise, rng.uniform(0.04, 0.1))
    if rng.random() < 0.3:  # uneven lighting
        shade = Image.linear_gradient("L").resize(img.size).rotate(rng.choice([0, 90, 180, 270]))
        img = Image.composite(img, Image.new("RGB", img.size, (0, 0, 0)), shade.point(lambda p: 255 - p // 4))
    return img


def canvas(rng: random.Random):
    w, h = rng.choice(SIZES)
    bg, ink, accent = palette(rng)
    img = Image.new("RGB", (w, h), bg)
    return img, ImageDraw.Draw(img), w, h, ink, accent


# --------------------------------------------------------------------------- #
# Date phrasing on images                                                      #
# --------------------------------------------------------------------------- #

def future(rng: random.Random, now: datetime, lo=1, hi=30) -> datetime:
    return now + timedelta(days=rng.randint(lo, hi))


def date_on_image(rng: random.Random, d: datetime) -> str:
    wd, mon = G.WEEKDAYS[d.weekday()], G.MONTHS[d.month - 1]
    return rng.choice([f"{wd}, {mon} {d.day}", f"{wd[:3].upper()} {mon[:3].upper()} {d.day}", f"{mon} {d.day}, {d.year}",
                       f"{d.month}/{d.day}/{d.year}", f"{d.day} {mon[:3]} {d.year}", f"{mon[:3]} {d.day}"])


def time_on_image(rng: random.Random, h: int, m: int) -> str:
    h12, ap = h % 12 or 12, "AM" if h < 12 else "PM"
    return rng.choice([f"{h12}:{m:02d} {ap}", f"{h12}:{m:02d}{ap.lower()}", f"{h:02d}:{m:02d}"] +
                      ([f"{h12} {ap}", f"{h12}{ap.lower()}"] if m == 0 else []))


def at(d: datetime, h: int, m: int) -> datetime:
    return d.replace(hour=h, minute=m, second=0, microsecond=0)


# --------------------------------------------------------------------------- #
# Image kinds. Each returns (image, facts) — facts drive the conversation.     #
# --------------------------------------------------------------------------- #

VENUES = ["SMX Convention Center", "UP Theater", "Room 302, Engineering Bldg", "The Ruins, Poblacion", "Zoom",
          "Café Lupe", "BGC Amphitheater", "Main Library, 3F", "Ramen Nagi, SM Aura", "Covered Court"]
POSTER_ORGS = ["", "", "CS Society", "Rotaract", "DevFest", "Barangay 12", "AIESEC", "Math Club", "UP Mountaineers",
               "Green Org", "Student Council", "Lions Club", "IEEE"]
POSTER_EVENTS = ["Career Fair", "Acoustic Night", "Hackathon", "Blood Donation Drive", "Film Screening", "Pitch Night",
                 "Fun Run", "General Assembly", "Python Workshop", "Art Exhibit", "Coastal Cleanup", "Quiz Bee",
                 "Open Mic", "Job Expo", "Design Talk", "Leadership Summit", "Tree Planting", "Book Fair", "Game Night",
                 "Photography Walk", "Concert for a Cause", "Seminar on AI", "Basketball Finals", "Bazaar"]
SUBJECTS = ["Calculus II", "Data Structures", "Physics 71", "Philippine History", "Technical Writing", "Statistics",
            "Operating Systems", "Discrete Math", "Chemistry Lab", "Economics"]
BILLERS = [("Meralco", "electricity"), ("PLDT Home", "internet"), ("Globe Postpaid", "phone"), ("Maynilad", "water"),
           ("Converge", "internet"), ("BPI Credit Card", "credit card")]


def img_poster(rng, now):
    img, d, w, h, ink, accent = canvas(rng)
    title = f"{rng.choice(POSTER_ORGS)} {rng.choice(POSTER_EVENTS)}".strip()
    day = future(rng, now, 2, 40)
    hh, mm = rng.choice([(9, 0), (10, 0), (13, 0), (14, 30), (17, 0), (18, 0), (19, 0), (19, 30)])
    venue = rng.choice(VENUES)
    d.rectangle([0, 0, w, rng.randint(8, 30)], fill=accent)
    y = int(h * rng.uniform(0.1, 0.2))
    tsize = rng.randint(34, 50)
    tf = font(rng, tsize, "bold")
    tcol = accent if rng.random() < 0.4 else ink
    for line in wrap(d, title.upper() if rng.random() < 0.5 else title, tf, w - 60):
        d.text((30, y), line, fill=tcol, font=tf)
        y += tsize + 12
    y += 20
    fs = rng.randint(22, 28)
    dstr, tstr = date_on_image(rng, day), time_on_image(rng, hh, mm)
    for line in [dstr, tstr, venue]:
        d.text((30, y), line, fill=ink, font=fit(d, rng, line, fs, w - 50))
        y += fs + 14
    if rng.random() < 0.6:
        d.text((30, h - 60), rng.choice(["Free entrance!", "Register at the door", "Open to all", "Bring a friend"]),
               fill=ink, font=font(rng, 18))
    for _ in range(rng.randint(0, 4)):  # decoration
        x0, y0 = rng.randint(w // 2, w), rng.randint(h // 2, h)
        d.ellipse([x0 - 40, y0 - 40, x0 + 40, y0 + 40], outline=accent, width=4)
    start = at(day, hh, mm)
    return img, {"kind": "poster", "title": title, "start": start, "venue": venue,
                 "text": f"{title}\n{dstr}\n{tstr}\n{venue}"}


def img_bill(rng, now):
    img, d, w, h, ink, accent = canvas(rng)
    biller, what = rng.choice(BILLERS)
    due = future(rng, now, 2, 25)
    amount = rng.choice([845.5, 1299, 1699, 2340.75, 3120, 4870.2, 999, 15500])
    acct = "".join(rng.choice("0123456789") for _ in range(10))
    cur = rng.choice(["PHP", "Php", "P"])  # many fonts lack the ₱ glyph
    d.rectangle([0, 0, w, 70], fill=accent)
    d.text((24, 18), biller, fill=(255, 255, 255), font=font(rng, 30, "bold"))
    rows = [("Account No.", acct), ("Billing period", f"{G.MONTHS[(due.month - 2) % 12][:3]} 1–{28}"),
            ("Amount due", f"{cur} {amount:,.2f}"), ("Due date", dstr := date_on_image(rng, due))]
    rng.shuffle(rows[:2])
    y = 110
    f = font(rng, 22 if w >= 500 else 18)
    vx = max(w // 2, int(max(d.textlength(k, font=f) for k, _ in rows)) + 44)
    for k, v in rows:
        d.text((24, y), k, fill=ink, font=f)
        d.text((vx, y), v, fill=ink, font=fit(d, rng, v, 22, w - vx - 12, "bold" if k in ("Amount due", "Due date") else "any"))
        y += 48
    d.line([24, y, w - 24, y], fill=ink, width=2)
    d.text((24, y + 16), "Pay on time to avoid disconnection.", fill=ink, font=font(rng, 16))
    return img, {"kind": "bill", "biller": biller, "what": what, "due": at(due, 23, 59), "amount": amount,
                 "text": f"{biller}\nAccount No. {acct}\nAmount due {cur} {amount:,.2f}\nDue date {dstr}"}


def img_assignment(rng, now):
    img, d, w, h, ink, accent = canvas(rng)
    subj = rng.choice(SUBJECTS)
    num = rng.randint(1, 9)
    kind = rng.choice(["Problem Set", "Homework", "Lab Report", "Worksheet", "Reading Quiz"])
    due = future(rng, now, 1, 14)
    hh, mm = rng.choice([(23, 59), (17, 0), (8, 0), (12, 0)])
    n_items = rng.randint(2, 5)
    items = [f"Question {i}" for i in range(1, n_items + 1)] if rng.random() < 0.6 else \
        rng.sample(["Read pages 40-55", "Answer review questions", "Draw the free-body diagram", "Show all solutions",
                    "Submit as PDF", "Cite 3 sources", "Write a 1-page summary"], n_items)
    d.text((24, 24), subj, fill=accent, font=fit(d, rng, subj, 26, w - 40, "bold"))
    d.text((24, 64), f"{kind} {num}", fill=ink, font=fit(d, rng, f"{kind} {num}", 30, w - 40, "bold"))
    due_line = f"Due: {date_on_image(rng, due)}, {time_on_image(rng, hh, mm)}"
    d.text((24, 112), due_line, fill=ink, font=fit(d, rng, due_line, 22, w - 40))
    y = 160
    for i, it in enumerate(items, 1):
        line = f"{i}. {it}" if not it.startswith("Question") else it
        d.text((36, y), line, fill=ink, font=fit(d, rng, line, 20, w - 50))
        y += 34
        if y > h - 40:
            items = items[:i]
            break
    title = f"{subj} {kind.lower()} {num}" if rng.random() < 0.5 else f"{subj} — {kind} {num}"
    return img, {"kind": "assignment", "title": title, "due": at(due, hh, mm), "items": items,
                 "text": f"{subj}\n{kind} {num}\n{due_line}\n" + "\n".join(items)}


def img_schedule(rng, now):
    img, d, w, h, ink, accent = canvas(rng)
    n = rng.randint(2, 4)
    days = rng.sample(range(5), n)
    rows = []
    for wd, subj in zip(sorted(days), rng.sample(SUBJECTS, n)):
        hh = rng.choice([7, 8, 9, 10, 13, 14, 15, 16])
        mm = rng.choice([0, 30])
        rows.append((wd, hh, mm, subj, f"Rm {rng.randint(100, 450)}", time_on_image(rng, hh, mm)))
    d.text((20, 20), rng.choice(["CLASS SCHEDULE", "My Schedule", "Weekly Timetable"]), fill=accent, font=font(rng, 28, "bold"))
    cols = [20, int(w * 0.24), int(w * 0.46), int(w * 0.82)]
    y = 80
    for txt, x in zip(["Day", "Time", "Subject", "Room"], cols):
        d.text((x, y), txt, fill=ink, font=font(rng, 18, "bold"))
    y += 34
    d.line([16, y - 6, w - 16, y - 6], fill=ink, width=2)
    fs = 17 if w < 500 else 19
    for wd, hh, mm, subj, room, tstr in rows:
        for txt, x, nx in zip([G.WEEKDAYS[wd][:3], tstr, subj, room], cols, cols[1:] + [w - 8]):
            d.text((x, y), txt, fill=ink, font=fit(d, rng, txt, fs, nx - x - 8))
        y += 40
        d.line([16, y - 8, w - 16, y - 8], fill=(160, 160, 160), width=1)
    events = [(subj, at(G.next_weekday(now, wd), hh, mm), room) for wd, hh, mm, subj, room, _ in rows]
    return img, {"kind": "schedule", "rows": events,
                 "text": "\n".join(f"{G.WEEKDAYS[wd][:3]} {t} {s} {r}" for wd, hh, mm, s, r, t in rows)}


def img_chat(rng, now):
    img, d, w, h, ink, accent = canvas(rng)
    who = rng.choice(G.PEOPLE[:15])
    plan = rng.choice(["dinner", "lunch", "coffee", "study session", "movie", "badminton", "videoke"])
    place = rng.choice(["Ramen Nagi", "Starbucks Katipunan", "SM North", "the library", "Gerry's Grill", "UP Town Center"])
    day_txt, day = rng.choice([("tomorrow", now + timedelta(days=1)), ("bukas", now + timedelta(days=1))] +
                              [(G.WEEKDAYS[i].lower(), G.next_weekday(now, i)) for i in range(7) if i != now.weekday()])
    hh = rng.choice([10, 12, 15, 17, 18, 19, 20])
    t_txt = f"{hh % 12 or 12}{'am' if hh < 12 else 'pm'}"
    msgs = [(who, rng.choice([f"{plan} {day_txt} {t_txt} at {place}?", f"tara {plan} {day_txt}, {t_txt} sa {place}",
                              f"hey! {plan} {day_txt} at {t_txt}? {place}"])),
            ("me", rng.choice(["sure!", "g!", "yes see you", "game"])),
            (who, rng.choice(["ok see you!", "nice, I'll reserve", "dont be late haha"]))]
    d.rectangle([0, 0, w, 56], fill=accent)
    d.text((18, 14), who, fill=(255, 255, 255), font=font(rng, 24, "bold"))
    y = 80
    f = font(rng, 19)
    for sender, txt in msgs:
        lines = wrap(d, txt, f, int(w * 0.6))
        bh = 16 + 26 * len(lines)
        bw = max(d.textlength(l, font=f) for l in lines) + 28
        x0 = 16 if sender != "me" else w - bw - 16
        fill = (228, 230, 235) if sender != "me" else accent
        d.rounded_rectangle([x0, y, x0 + bw, y + bh], radius=16, fill=fill)
        for i, l in enumerate(lines):
            d.text((x0 + 14, y + 8 + 26 * i), l, fill=(20, 20, 20) if sender != "me" else (255, 255, 255), font=f)
        y += bh + 14
    title = f"{plan.capitalize()} with {who}"
    return img, {"kind": "chat", "title": title, "start": at(day, hh, 0), "venue": place, "who": who,
                 "text": "\n".join(f"{s}: {t}" for s, t in msgs)}


def img_note(rng, now):
    bg = rng.choice([(255, 244, 150), (255, 200, 210), (190, 230, 255), (210, 245, 200), (255, 255, 255)])
    w, h = rng.choice(SIZES)
    img = Image.new("RGB", (w, h), bg)
    d = ImageDraw.Draw(img)
    if rng.random() < 0.3:
        title, body, _ = rng.choice(G.NOTE_TOPICS)
        items = [s.strip() for s in body.replace(";", ",").replace(". ", ", ").split(",") if s.strip()][:6]
    else:
        title, pool = rng.choice(NOTE_LISTS)
        items = rng.sample(pool, rng.randint(2, min(6, len(pool))))
    if rng.random() < 0.5:
        for yy in range(70, h, 36):
            d.line([0, yy, w, yy], fill=(170, 190, 220), width=1)
    f = font(rng, rng.randint(22, 28), "hand")
    d.text((26, 22), title, fill=(30, 30, 90), font=font(rng, 30, "hand"))
    y = 76
    for it in items:
        for line in wrap(d, ("• " if rng.random() < 0.6 else "- ") + it, f, w - 50):
            d.text((30, y), line, fill=(30, 30, 90), font=f)
            y += 36
    return img, {"kind": "note", "title": title, "items": items, "text": title + "\n" + "\n".join(items)}


NOTE_LISTS = [
    ("Groceries", ["eggs", "milk", "rice 5kg", "bread", "coffee", "bananas", "chicken", "soap", "toothpaste", "onions", "garlic", "tuyo"]),
    ("Packing list", ["passport", "charger", "toothbrush", "3 shirts", "jacket", "umbrella", "meds", "slippers", "power bank"]),
    ("Ideas", ["offline habit tracker", "P2P grocery list", "study timer widget", "voice notes to tasks", "budget bot"]),
    ("To buy", ["HDMI cable", "notebook", "ballpen", "printer ink", "extension cord", "phone case"]),
    ("Meeting notes", ["ship v2 Friday", "Josh owns QA", "Mia: release notes", "budget approved", "next sync Monday"]),
    ("Reviewer topics", ["recursion", "linked lists", "Big-O", "hash tables", "BFS vs DFS", "heaps"]),
    ("Weekend", ["laundry", "call Mom", "clean room", "jog 5k", "meal prep", "fix bike"]),
    ("Movies to watch", ["Past Lives", "Spirited Away", "Dune Part Two", "Heneral Luna", "Coco", "Interstellar"]),
    ("Gift ideas", ["orchid plant", "scarf", "cooking class", "wireless earbuds", "photo album", "mug"]),
    ("Budget", ["rent 12000", "food 6000", "transport 2000", "savings 5000", "load 300", "netflix 549"]),
]
SHAPES = ["circle", "square", "triangle"]
COLORS = {"red": (220, 40, 40), "blue": (40, 90, 220), "green": (40, 170, 70), "yellow": (240, 210, 30),
          "orange": (245, 140, 20), "purple": (140, 60, 190), "black": (15, 15, 15)}


def img_shapes(rng, now):
    w, h = rng.choice(SIZES)
    img = Image.new("RGB", (w, h), (255, 255, 255) if rng.random() < 0.7 else (235, 235, 235))
    d = ImageDraw.Draw(img)
    n = rng.randint(1, 4)
    objs = []
    for i in range(n):
        shape, (cname, col) = rng.choice(SHAPES), rng.choice(list(COLORS.items()))
        cx, cy = int(w * (i + 0.5) / n), rng.randint(h // 3, 2 * h // 3)
        r = min(w // (2 * n), 90) - 6
        if shape == "circle":
            d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=col)
        elif shape == "square":
            d.rectangle([cx - r, cy - r, cx + r, cy + r], fill=col)
        else:
            d.polygon([(cx, cy - r), (cx - r, cy + r), (cx + r, cy + r)], fill=col)
        objs.append((cname, shape))
    return img, {"kind": "shapes", "objs": objs, "text": ""}


KINDS = [(16, img_poster), (14, img_bill), (16, img_assignment), (12, img_schedule), (14, img_chat), (14, img_note),
         (6, img_shapes)]


# --------------------------------------------------------------------------- #
# Conversations about an image                                                 #
# --------------------------------------------------------------------------- #

def ocr_noise(rng: random.Random, text: str) -> str:
    """The app runs ML Kit OCR on attachments and sends the text too; imitate its slips."""
    swaps = {"0": "O", "1": "l", "5": "S", "rn": "m", "₱": "P"}
    for a, b in swaps.items():
        if rng.random() < 0.15:
            text = text.replace(a, b, 1)
    return text


def user_turn(rng, text: str, facts: dict, image_name: str) -> dict:
    text = G.user(text, rng)["content"]  # the app never sends an attachment without typed text
    if facts["text"] and rng.random() < 0.4:  # app also attaches OCR text (see agent.tsx → extractedContext)
        text = f"{text}\n\n[Attached content]\n[Image: {image_name}]\n{ocr_noise(rng, facts['text'])}".strip()
    parts = [{"type": "image"}]
    if text:
        parts.append({"type": "text", "text": text})
    return {"role": "user", "content": parts}


def convo(rng: random.Random, c: G.Ctx, facts: dict, name: str) -> list[dict]:
    k, now = facts["kind"], c.now
    ask_only = rng.random() < 0.2 and k != "shapes"

    if k == "shapes":
        objs = facts["objs"]
        q, a = rng.choice([
            ("what's in this picture?", ", ".join(f"a {col} {sh}" for col, sh in objs).capitalize() + "."),
            ("how many shapes are there?", f"There {'is' if len(objs) == 1 else 'are'} {len(objs)}."),
            ("what color is the first shape?", f"It's {objs[0][0]}."),
        ])
        return [user_turn(rng, q, facts, name), G.say(a)]

    if ask_only:  # read or answer about the image, no tool
        if k == "bill":
            q, a = rng.choice([("when is this due?", f"It's due {G.human_day(facts['due'], now)}."),
                               ("how much do I owe?", f"₱{facts['amount']:,.2f} to {facts['biller']}, due {G.human_day(facts['due'], now)}.")])
        elif k in ("poster", "chat"):
            s = facts["start"]
            q, a = ("when is this?", f"{G.cap(G.human_day(s, now))} at {G.human_time(s)}, at {facts['venue']}.")
        elif k == "assignment":
            q, a = ("what's this about?", f"It's {facts['title']}, due {G.human_day(facts['due'], now)} at {G.human_time(facts['due'])}, with {len(facts['items'])} items.")
        else:
            q, a = rng.choice(["what does this say?", "can you read this for me?", "transcribe this"]), facts["text"].replace("\n", "; ") if k != "note" else facts["title"] + ": " + ", ".join(facts["items"]) + "."
        return [user_turn(rng, q, facts, name), G.say(a)]

    if k in ("poster", "chat"):
        s = facts["start"]
        q = rng.choice(["add this to my calendar", "save this event", "schedule this", "put this in my calendar",
                        "pa-add nito sa calendar", "remind me about this", "add it"])
        args = {"title": facts["title"], "startDate": G.iso(s)}
        if rng.random() < 0.6:
            args["description"] = facts["venue"]
        call = c.call("add_event", args)
        return [user_turn(rng, q, facts, name), G.assistant_calls(call),
                G.tool_msg(call, f"Added event: \"{facts['title']}\" on {G.locale_string(s)}"),
                G.say(f"Added \"{facts['title']}\" {G.human_day(s, now)} at {G.human_time(s)}" + (f" at {facts['venue']}." if "description" in args else "."))]

    if k == "bill":
        title = f"Pay {facts['biller']} bill"
        persistent = rng.random() < 0.5
        q = rng.choice(["remind me to pay this", "add this bill to my tasks", "don't let me forget this bill",
                        "remind me until I pay this"] if persistent else ["remind me to pay this", "add this bill to my tasks", "add this"])
        persistent = persistent and ("until" in q or "forget" in q)
        args = {"title": title, "dueDate": G.iso(facts["due"])}
        if persistent:
            args["reminder"] = "persistent"
        call = c.call("add_todo", args)
        res = f"Added task \"{title}\" due {G.short_due(facts['due'])}." if not persistent else \
            f"Added task \"{title}\" due {G.short_due(facts['due'])}. I'll keep reminding until it's done."
        return [user_turn(rng, q, facts, name), G.assistant_calls(call), G.tool_msg(call, res),
                G.say(f"Added \"{title}\" (₱{facts['amount']:,.2f}), due {G.human_day(facts['due'], now)}." +
                      (" I'll keep reminding you until it's paid." if persistent else ""))]

    if k == "assignment":
        q = rng.choice(["add this homework", "add this to my tasks", "remind me about this assignment",
                        "make a task with each question", "add this, keep reminding me until done"])
        args = {"title": facts["title"], "dueDate": G.iso(facts["due"])}
        if "question" in q or rng.random() < 0.5:
            args["items"] = facts["items"]
        if "keep reminding" in q:
            args["reminder"] = "persistent"
        call = c.call("add_todo", args)
        res = f"Added task \"{facts['title']}\" due {G.short_due(facts['due'])}"
        if "items" in args:
            res += f" with {len(facts['items'])} checklist item{'s' if len(facts['items']) > 1 else ''}"
        res += ". I'll keep reminding until it's done." if "reminder" in args else "."
        return [user_turn(rng, q, facts, name), G.assistant_calls(call), G.tool_msg(call, res),
                G.say(f"Added \"{facts['title']}\", due {G.human_day(facts['due'], now)} at {G.human_time(facts['due'])}" +
                      (f" with {len(facts['items'])} checklist items." if "items" in args else "."))]

    if k == "schedule":
        q = rng.choice(["add my classes to the calendar", "put this schedule in my calendar", "add these classes this week"])
        calls = [c.call("add_event", {"title": s, "startDate": G.iso(d), "description": r}) for s, d, r in facts["rows"]]
        return [user_turn(rng, q, facts, name), G.assistant_calls(*calls),
                *[G.tool_msg(cl, f"Added event: \"{s}\" on {G.locale_string(d)}") for cl, (s, d, _) in zip(calls, facts["rows"])],
                G.say(f"Added {len(calls)} classes: " + ", ".join(f"{s} {G.WEEKDAYS[d.weekday()][:3]} {G.human_time(d)}" for s, d, _ in facts["rows"]) + ".")]

    # note
    q = rng.choice(["save this as a note", "write this down", "note this", "save this"])
    args = {"title": facts["title"], "body": "\n".join(facts["items"])}
    call = c.call("add_note", args)
    return [user_turn(rng, q, facts, name), G.assistant_calls(call), G.tool_msg(call, f"Added note: \"{facts['title']}\""),
            G.say(f"Saved \"{facts['title']}\" to your notes.")]


def make_vision_example(rng: random.Random, tools: list[dict], img_dir: Path, idx: str) -> dict:
    now = datetime(2026, 1, 1) + timedelta(days=rng.randint(0, 729), hours=rng.randint(6, 22), minutes=rng.randint(0, 59))
    c = G.Ctx(rng, now)
    fn = rng.choices([f for _, f in KINDS], weights=[w for w, _ in KINDS])[0]
    img, facts = fn(rng, now)
    if facts["kind"] != "shapes":
        img = photo_effects(img, rng)
    path = img_dir / f"{idx}.jpg"
    img.save(path, quality=rng.randint(70, 92))
    msgs = convo(rng, c, facts, rng.choice(["photo", "IMG_2041.jpg", "screenshot", "image"]))
    return {"scenario": "vision_" + facts["kind"], "images": [str(path.name)],
            "messages": [{"role": "system", "content": G.system_prompt(now, G.known_context(rng))}] + msgs,
            "tools": tools}


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--train", type=int, default=1600, help="image conversations")
    ap.add_argument("--eval", type=int, default=160)
    ap.add_argument("--text-ratio", type=float, default=0.4, help="extra text-only conversations, as a share of the image ones")
    ap.add_argument("--seed", type=int, default=11)
    ap.add_argument("--tools", default=str(HERE / "tools.json"))
    ap.add_argument("--out", default=str(HERE / "data_vision"))
    a = ap.parse_args()
    tools = json.loads(Path(a.tools).read_text())
    out = Path(a.out)
    (out / "images").mkdir(parents=True, exist_ok=True)
    for split, n, seed in (("train", a.train, a.seed), ("eval", a.eval, a.seed + 1000)):
        rng = random.Random(seed)
        rows = [make_vision_example(rng, tools, out / "images", f"{split}-{i:05d}") for i in range(n)]
        rows += [G.make_example(rng, tools) for _ in range(math.ceil(n * a.text_ratio))]
        rng.shuffle(rows)
        with open(out / f"{split}.jsonl", "w") as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"{split}: {len(rows)} conversations ({n} with images) → {out / f'{split}.jsonl'}")


if __name__ == "__main__":
    main()

In [ ]:
!python3 generate_vision_dataset.py --train $TRAIN_N --eval $EVAL_N --seed 11 --out data_vision
import json
from PIL import Image
from IPython.display import display

def load_rows(path, img_dir):
    rows = [json.loads(l) for l in open(path)]
    for r in rows:
        r["image_paths"] = [f"{img_dir}/{p}" for p in r.get("images", [])]
    return rows

train_rows = load_rows("data_vision/train.jsonl", "data_vision/images")
eval_rows  = load_rows("data_vision/eval.jsonl", "data_vision/images")
ex = next(r for r in train_rows if r["image_paths"])
display(Image.open(ex["image_paths"][0]).resize((256, 256)))
print(json.dumps(ex["messages"][1:], indent=1, ensure_ascii=False)[:1500])

## 2 · Model

In [ ]:
import torch
from unsloth import FastVisionModel
model, processor = FastVisionModel.from_pretrained(BASE_MODEL, load_in_4bit=False, use_gradient_checkpointing="unsloth")
model = FastVisionModel.get_peft_model(
    model, finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=7,
)
tok = processor.tokenizer if hasattr(processor, "tokenizer") else processor
CHAT_TEMPLATE = getattr(processor, "chat_template", None) or tok.chat_template
MAX_LEN = 4096  # the app runs llama.rn with n_ctx 4096

def render(row, upto=None, gen_prompt=False):
    # Rendered with plain dicts (not a datasets.Dataset), so tool-call arguments keep exactly their own keys.
    msgs = row["messages"] if upto is None else row["messages"][:upto]
    return tok.apply_chat_template(msgs, tools=row["tools"], chat_template=CHAT_TEMPLATE,
                                   tokenize=False, add_generation_prompt=gen_prompt)

def images_of(row):
    return [Image.open(p).convert("RGB") for p in row["image_paths"]]

ASSIST = tok.encode("<|im_start|>assistant\n", add_special_tokens=False)
IM_END = tok.convert_tokens_to_ids("<|im_end|>")

def assistant_labels(ids):
    # Learn only assistant turns (tool calls + replies); system, user, images and tool results are masked.
    labels, i, n = [-100] * len(ids), 0, len(ASSIST)
    while i < len(ids):
        if ids[i:i + n] == ASSIST:
            j = i + n
            while j < len(ids) and ids[j] != IM_END:
                j += 1
            labels[i + n:j + 1] = ids[i + n:j + 1]
            i = j
        i += 1
    return labels

def collate(batch):
    texts, imgs = [render(r) for r in batch], [im for r in batch for im in images_of(r)]
    enc = processor(text=texts, images=imgs or None, return_tensors="pt", padding=True)
    labels = torch.tensor([assistant_labels(x.tolist()) for x in enc["input_ids"]])
    labels[enc["attention_mask"] == 0] = -100
    enc["labels"] = labels
    return enc

b = collate([ex])
print("tokens:", b["input_ids"].shape[1], "| trained tokens:", int((b["labels"] != -100).sum()))
print(tok.decode([t for t in b["labels"][0].tolist() if t != -100]))

## 3 · Accuracy check (generates, parses `<tool_call>` blocks, compares to ground truth)

In [ ]:
import re
CALL_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)

def parse_calls(text):
    out = []
    for m in CALL_RE.finditer(text):
        try:
            c = json.loads(m.group(1)); out.append((c.get("name"), c.get("arguments") or {}))
        except json.JSONDecodeError:
            out.append(("<bad json>", {}))
    return out

def norm(v):
    if isinstance(v, str): return v.strip().lower()
    if isinstance(v, list): return [norm(x) for x in v]
    return v

def same_args(a, b):
    return isinstance(a, dict) and set(a) == set(b) and all(norm(a[k]) == norm(b[k]) for k in b)

@torch.no_grad()
def evaluate(rows, n=EVAL_SAMPLES, batch=8, show=4):
    FastVisionModel.for_inference(model); tok.padding_side = "left"
    cases = [(r, i, m) for r in rows[:n] for i, m in enumerate(r["messages"]) if m["role"] == "assistant"]
    st = {k: 0 for k in ("tool", "name", "exact", "none", "none_ok", "img_tool", "img_exact")}
    misses = []
    for s in range(0, len(cases), batch):
        chunk = cases[s:s + batch]
        enc = processor(text=[render(r, upto=i, gen_prompt=True) for r, i, _ in chunk],
                        images=[im for r, _, _ in chunk for im in images_of(r)] or None,
                        return_tensors="pt", padding=True).to("cuda")
        gen = model.generate(**enc, max_new_tokens=320, do_sample=False, pad_token_id=tok.pad_token_id)
        for (r, i, want), g in zip(chunk, gen):
            text = tok.decode(g[enc["input_ids"].shape[1]:], skip_special_tokens=False).split("<|im_end|>")[0]
            got = parse_calls(text)
            exp = [(c["function"]["name"], c["function"]["arguments"]) for c in want.get("tool_calls", [])]
            if exp:
                names = [x[0] for x in got] == [e[0] for e in exp]
                ok = names and all(same_args(x[1], e[1]) for x, e in zip(got, exp))
                st["tool"] += 1; st["name"] += names; st["exact"] += ok
                if r["image_paths"]: st["img_tool"] += 1; st["img_exact"] += ok
            else:
                ok = not got; st["none"] += 1; st["none_ok"] += ok
            if not ok and len(misses) < show:
                misses.append((r["scenario"], exp or want["content"], text.strip()[:300]))
    res = {"tool_name_acc": st["name"] / max(st["tool"], 1), "tool_exact_acc": st["exact"] / max(st["tool"], 1),
           "image_exact_acc": st["img_exact"] / max(st["img_tool"], 1), "no_tool_acc": st["none_ok"] / max(st["none"], 1)}
    print({k: round(v, 3) for k, v in res.items()})
    for scen, exp, got in misses:
        print(f"  ✗ [{scen}] expected {exp}\n    got {got!r}")
    FastVisionModel.for_training(model); tok.padding_side = "right"
    return res

print("Base model, before training:")
base_scores = evaluate(eval_rows, n=40)

## 4 · Train until done

In [ ]:
import subprocess
from dataclasses import fields
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

_cfg = {f.name for f in fields(SFTConfig)}
LEN_ARG = "max_seq_length" if "max_seq_length" in _cfg else "max_length"  # renamed in newer TRL
bf16 = torch.cuda.is_bf16_supported()

def make_trainer(train, evals, epochs, lr):
    FastVisionModel.for_training(model)
    return SFTTrainer(
        model=model, processing_class=processor, data_collator=collate,
        train_dataset=train, eval_dataset=evals,
        args=SFTConfig(
            per_device_train_batch_size=2, gradient_accumulation_steps=8, per_device_eval_batch_size=2,
            num_train_epochs=epochs, learning_rate=lr, lr_scheduler_type="cosine", warmup_ratio=0.05,
            weight_decay=0.01, optim="adamw_8bit", seed=7, fp16=not bf16, bf16=bf16,
            logging_steps=5, eval_strategy="steps", eval_steps=25, save_strategy="steps", save_steps=25,
            save_total_limit=2, load_best_model_at_end=True, metric_for_best_model="eval_loss",
            output_dir="checkpoints", report_to="none",
            remove_unused_columns=False, dataset_text_field="", dataset_kwargs={"skip_prepare_dataset": True},
            **{LEN_ARG: MAX_LEN},
        ),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)],
    )

eval_loss_rows = eval_rows[:80]
history = []
for rnd in range(1, MAX_ROUNDS + 1):
    if rnd > 1:  # fresh, unseen photos and conversations
        subprocess.run(["python3", "generate_vision_dataset.py", "--train", str(TRAIN_N // 2), "--eval", "8",
                        "--seed", str(100 + rnd), "--out", f"data_vision_r{rnd}"], check=True)
        train_rows = load_rows(f"data_vision_r{rnd}/train.jsonl", f"data_vision_r{rnd}/images")
    print(f"\n══ Round {rnd}: {len(train_rows)} conversations ══")
    make_trainer(train_rows, eval_loss_rows, MAX_EPOCHS if rnd == 1 else 1, 2e-4 if rnd == 1 else 5e-5).train()
    scores = evaluate(eval_rows)
    history.append(scores)
    if scores["tool_exact_acc"] >= TARGET_ACC and scores["no_tool_acc"] >= 0.85:
        print(f"Target reached in round {rnd}."); break
else:
    print("Round limit reached — last scores:", history[-1], "(raise MAX_ROUNDS to keep going)")

In [ ]:
# Try it on a fresh photo
import random, datetime, generate_vision_dataset as V
rng = random.Random(2024); now = datetime.datetime.now().replace(second=0, microsecond=0)
img, facts = V.img_poster(rng, now); img = V.photo_effects(img, rng); img.save("try.jpg"); display(img.resize((256, 256)))
row = {"tools": eval_rows[0]["tools"], "image_paths": ["try.jpg"], "messages": [
    {"role": "system", "content": V.G.system_prompt(now)},
    {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "add this to my calendar"}]}]}
FastVisionModel.for_inference(model)
enc = processor(text=[render(row, gen_prompt=True)], images=images_of(row), return_tensors="pt").to("cuda")
out = model.generate(**enc, max_new_tokens=200, do_sample=False)
print(tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=False))
print("truth:", facts["title"], facts["start"], facts["venue"])

## 5 · Export: model GGUF + vision projector (mmproj) for llama.rn

In [ ]:
model.save_pretrained(f"{OUT_NAME}-lora"); processor.save_pretrained(f"{OUT_NAME}-lora")
model.save_pretrained_merged("merged", processor, save_method="merged_16bit")

In [ ]:
%%bash
set -e
[ -d llama.cpp ] || git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
pip install -q ./llama.cpp/gguf-py
python llama.cpp/convert_hf_to_gguf.py merged --outfile seishin-vision-F16.gguf --outtype f16
python llama.cpp/convert_hf_to_gguf.py merged --mmproj --outfile mmproj-seishin-vision-Q8_0.gguf --outtype q8_0
cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /dev/null
cmake --build llama.cpp/build --target llama-quantize llama-server -j "$(nproc)" > /dev/null

In [ ]:
import glob, os, shutil, subprocess
GGUF = f"{OUT_NAME}-{QUANT}.gguf"
subprocess.run(["llama.cpp/build/bin/llama-quantize", f"{OUT_NAME}-F16.gguf", GGUF, QUANT], check=True)
MMPROJ = sorted(glob.glob("*mmproj*.gguf"))[0]
for f in (GGUF, MMPROJ):
    print(f, f"{os.path.getsize(f) / 1e6:.0f} MB")

### Smoke test — llama.cpp server with `--jinja --mmproj` (the path llama.rn uses), real image + tools

In [ ]:
import base64, time, requests
proc = subprocess.Popen(["llama.cpp/build/bin/llama-server", "-m", GGUF, "--mmproj", MMPROJ, "--jinja",
                         "-c", "4096", "--port", "8089"],
                        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
try:
    for _ in range(120):
        try:
            if requests.get("http://127.0.0.1:8089/health", timeout=1).ok: break
        except requests.RequestException: pass
        time.sleep(1)
    tools = json.load(open("tools.json"))
    def chat(content):
        r = requests.post("http://127.0.0.1:8089/v1/chat/completions", timeout=600, json={
            "messages": [{"role": "system", "content": V.G.system_prompt(now)}, {"role": "user", "content": content}],
            "tools": tools, "tool_choice": "auto", "temperature": 0.3}).json()
        m = r["choices"][0]["message"]
        return [(c["function"]["name"], c["function"]["arguments"]) for c in m.get("tool_calls") or []], m.get("content")
    for kind in (V.img_bill, V.img_assignment, V.img_chat):
        img, facts = kind(rng, now); img.save("smoke.jpg")
        url = "data:image/jpeg;base64," + base64.b64encode(open("smoke.jpg", "rb").read()).decode()
        print(facts["kind"], "→", chat([{"type": "image_url", "image_url": {"url": url}}, {"type": "text", "text": "add this"}]))
    print("text →", chat("add team standup tomorrow at 9am"))
finally:
    proc.terminate()

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/seishin", exist_ok=True)
    for f in (GGUF, MMPROJ):
        shutil.copy(f, "/content/drive/MyDrive/seishin/")
    shutil.copytree(f"{OUT_NAME}-lora", f"/content/drive/MyDrive/seishin/{OUT_NAME}-lora", dirs_exist_ok=True)
    print("Saved to Drive: MyDrive/seishin/", GGUF, MMPROJ)
from google.colab import files
files.download(MMPROJ)
files.download(GGUF)   # ~1.1 GB — if the browser download stalls, grab it from Drive instead

## On the phone
1. Copy both files to the device (Drive app → download, or USB).
2. Seishin → **Settings → AI · on-device first → On-device model** → pick `seishin-vision-Q4_K_M.gguf` → Save.
3. Below it, **Vision projector** → pick `mmproj-seishin-vision-Q8_0.gguf`.
4. In the Agent tab, attach a photo and type e.g. *"add this to my calendar"*.

Needs ~1.8 GB free RAM while loaded. Without the projector the same model still works text-only.